In [1]:
!pip install mlflow dagshub optuna imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2

In [33]:
#!aws configure
import dagshub
dagshub.init(repo_owner="rohitbedse",repo_name="yt-comment-sentiment-analysis",mlflow=True)

⠸ Waiting for authorization

Accessing as rohitbedse

Initialized MLflow to track repo "rohitbedse/yt-comment-sentiment-analysis"

Repository rohitbedse/yt-comment-sentiment-analysis initialized!

In [34]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow")

In [35]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning With Chnaging Features")

<Experiment: artifact_location='mlflow-artifacts:/54b3eaefe0cb4e4d8246724361152697', creation_time=1775572221042, experiment_id='9', last_update_time=1775572221042, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning With Chnaging Features', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [36]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report,f1_score
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
import numpy as np

In [37]:
df = pd.read_csv('/content/reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [38]:
# -----------------------------
# STEP 1: Label Mapping
# -----------------------------
df = df.dropna(subset=['category'])
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# -----------------------------
# STEP 2: Show ORIGINAL label distribution
# -----------------------------
print("🔥 ORIGINAL LABELS:")
print(np.unique(df['category'], return_counts=True))

# -----------------------------
# STEP 3: TRAIN-TEST SPLIT (FIRST)
# -----------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# -----------------------------
# STEP 4: TF-IDF (FIT ONLY ON TRAIN)
# -----------------------------
vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=5000)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# -----------------------------
# STEP 5: Show TRAIN distribution BEFORE SMOTE
# -----------------------------
print("\n📊 TRAIN LABELS BEFORE SMOTE:")
print(np.unique(y_train, return_counts=True))

# -----------------------------
# STEP 6: APPLY SMOTE ONLY ON TRAIN
# -----------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# -----------------------------
# STEP 7: Show TRAIN distribution AFTER SMOTE
# -----------------------------
print("\n🚀 TRAIN LABELS AFTER SMOTE:")
print(np.unique(y_train_res, return_counts=True))

# -----------------------------
# STEP 9: MLflow logging
# -----------------------------
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Bigram")
        mlflow.set_tag("experiment", "No_Leakage_Pipeline")

        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("resampling", "SMOTE")
        mlflow.log_param("vectorizer", "TF-IDF Bigram")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        mlflow.sklearn.log_model(model, f"{model_name}_model")

# ================================
# Step 10: Optuna Objective (F1 Macro)
# ================================
def objective_logistic(trial):
    C = trial.suggest_float('C', 1e-3, 10.0, log=True)
    solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear'])

    model = LogisticRegression(
        C=C,
        solver=solver,
        max_iter=2000,   # 🔥 increase for convergence
        random_state=42
    )

    # ✅ SAME FLOW → split AFTER SMOTE
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_res,
        y_train_res,
        test_size=0.2,
        random_state=42,
        stratify=y_train_res
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)

    return f1_score(y_val, y_pred, average='macro')


# ================================
# Step 11: Run Optuna + Log Best Model
# ================================
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_logistic, n_trials=30)

    best_params = study.best_params
    print("\n🏆 BEST PARAMS:", best_params)

    best_model = LogisticRegression(
        C=best_params['C'],
        solver=best_params['solver'],
        max_iter=2000,
        random_state=42,
        multi_class='auto'
    )

    # ✅ FINAL TRAINING ON FULL SMOTE DATA
    log_mlflow(
        "LogisticRegression",
        best_model,
        X_train_res,   # full SMOTE train
        X_test,        # untouched test
        y_train_res,
        y_test
    )


# ================================
# RUN
# ================================
run_optuna_experiment()

🔥 ORIGINAL LABELS:
(array([0, 1, 2]), array([12644, 15770,  8248]))

📊 TRAIN LABELS BEFORE SMOTE:
(array([0, 1, 2]), array([10115, 12616,  6598]))


[I 2026-04-07 18:26:56,394] A new study created in memory with name: no-name-737cae4e-1aa3-460b-a7c5-208c8396a04c



🚀 TRAIN LABELS AFTER SMOTE:
(array([0, 1, 2]), array([12616, 12616, 12616]))


[I 2026-04-07 18:26:56,727] Trial 0 finished with value: 0.6311370982904269 and parameters: {'C': 0.0026275068220646606, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.6311370982904269.
[I 2026-04-07 18:26:57,350] Trial 1 finished with value: 0.638933106975614 and parameters: {'C': 0.0050223982329950225, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.638933106975614.
[I 2026-04-07 18:26:58,399] Trial 2 finished with value: 0.693338898505886 and parameters: {'C': 0.03673902671462507, 'solver': 'lbfgs'}. Best is trial 2 with value: 0.693338898505886.
[I 2026-04-07 18:27:01,819] Trial 3 finished with value: 0.8218459230393851 and parameters: {'C': 2.432142112647285, 'solver': 'lbfgs'}. Best is trial 3 with value: 0.8218459230393851.
[I 2026-04-07 18:27:02,089] Trial 4 finished with value: 0.6394384062637876 and parameters: {'C': 0.007976714396725022, 'solver': 'liblinear'}. Best is trial 3 with value: 0.8218459230393851.
[I 2026-04-07 18:27:02,284] Trial 5 finished with value: 0.633


🏆 BEST PARAMS: {'C': 9.431229106987209, 'solver': 'liblinear'}


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2026/04/07 18:27:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 18:28:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogisticRegression_SMOTE_TFIDF_Bigram at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9/runs/6b6efa0fa0424fd2aa1d6cc49b0c17ae
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9


In [39]:
# ✅ Best params (example from Optuna)
best_params = {
    'C': 9.43,            # replace with your best     # or 'l1'
}

# ✅ Model
model = LogisticRegression(
    **best_params,
    solver='liblinear',          # needed for l1/l2
    class_weight='balanced',     # 🔥 important for imbalance
    random_state=42,
    max_iter=2000                # avoid convergence issues
)

# ✅ Train (use SMOTE training data if applied)
model.fit(X_train_res, y_train_res)

# ✅ Predict
y_pred = model.predict(X_test)

# ✅ Evaluation
print("Test Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("Test Weighted F1:", f1_score(y_test, y_pred, average='weighted'))

# 🔥 Detailed report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Test Macro F1: 0.8202248899029492
Test Weighted F1: 0.8353249890872354

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.87      0.86      2529
           1       0.89      0.85      0.87      3154
           2       0.71      0.75      0.73      1650

    accuracy                           0.83      7333
   macro avg       0.82      0.82      0.82      7333
weighted avg       0.84      0.83      0.84      7333



In [40]:
import numpy as np

print("Unique predictions:", np.unique(y_pred))
print("Unique true labels:", np.unique(y_test))

Unique predictions: [0 1 2]
Unique true labels: [0 1 2]
